# DEAI-opdrachten clustering

imports

In [52]:
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

In [53]:
df = pd.read_excel("AmesHousing.xlsx", sheet_name="AmesHousing")

logboek = pd.DataFrame(columns=[
    "Experiment",
    "Features",
    "Hyperparameters",
    "Silhouette score",
    "Verwachting",
    "Resultaat",
    "Conclusie"
])

In [ ]:
def run_experiment(exp_name, features, cat_cols, n_clusters, use_scaling,
                   verwachting, resultaat, conclusie):

    df_exp = df.copy()

    # missing values oplossen
    df_exp = df_exp.fillna(df_exp.median(numeric_only=True))

    # one-hot encoding
    if len(cat_cols) > 0:
        df_exp = pd.get_dummies(df_exp, columns=cat_cols, drop_first=True)

    # encoded columns
    encoded_cols = [col for col in df_exp.columns if any(cat in col for cat in cat_cols)]

    # features samenstellen
    X = df_exp[[c for c in features if c not in cat_cols] + encoded_cols]

    # scaling
    if use_scaling:
        scaler = StandardScaler()
        X = scaler.fit_transform(X)

    # model
    kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
    labels = kmeans.fit_predict(X)
    centroids = kmeans.cluster_centers_


    # score
    score = silhouette_score(X, labels)

    # clusters toevoegen
    df_exp["Cluster"] = labels

    # logboek
    logboek.loc[len(logboek)] = {
        "Experiment": exp_name,
        "Features": ", ".join(features),
        "Hyperparameters": f"n_clusters={n_clusters}, n_init=10, scaling={use_scaling}",
        "Silhouette score": score,
        "Verwachting": verwachting,
        "Resultaat": resultaat,
        "Conclusie": conclusie
    }

    # output
    print(f"{exp_name} - silhouette score:", score)
    print(df_exp["Cluster"].value_counts())
    



In [55]:
run_experiment(
    "Exp 1",
    ["Gr Liv Area", "Overall Qual", "Neighborhood"],
    ["Neighborhood"],
    3,
    True,
    "Baseline clustering op grootte, kwaliteit en locatie",
    "Redelijke clustering met overlap",
    "Baseline werkt maar kan beter"
)

run_experiment(
    "Exp 2",
    ["Gr Liv Area", "Overall Qual", "Year Built", "Lot Area", "Neighborhood"],
    ["Neighborhood"],
    3,
    True,
    "Meer features verbeteren scheiding",
    "Lichte verbetering",
    "Extra features helpen beperkt"
)

run_experiment(
    "Exp 3",
    ["Gr Liv Area", "Overall Qual", "Year Built", "Lot Area", "Neighborhood"],
    ["Neighborhood"],
    5,
    True,
    "Meer clusters geven fijnere verdeling",
    "Clusters worden minder stabiel",
    "Te veel clusters vermindert kwaliteit"
)

run_experiment(
    "Exp 4",
    ["Gr Liv Area", "Overall Qual", "Neighborhood"],
    ["Neighborhood"],
    3,
    False,
    "Zonder scaling domineren grote waarden",
    "Slechtere clustering",
    "Scaling is essentieel"
)

run_experiment(
    "Exp 5",
    ["Gr Liv Area", "Overall Qual"],
    [],
    3,
    True,
    "Minder features geven minder ruis",
    "Minder complexe clusters",
    "Categorische features zijn belangrijk"
)

run_experiment(
    "Exp 6",
    ["Gr Liv Area", "Overall Qual", "House Style"],
    ["House Style"],
    3,
    True,
    "Andere categorische feature beïnvloedt clustering",
    "Andere clusterstructuur zichtbaar",
    "Feature-keuze is belangrijk"
)

run_experiment(
    "Exp 7",
    ["Gr Liv Area", "Overall Qual", "Neighborhood"],
    ["Neighborhood"],
    7,
    True,
    "Meer clusters = meer detail",
    "Clusters worden instabiel",
    "Te veel clusters is slecht"
)

run_experiment(
    "Exp 8",
    ["Gr Liv Area", "Overall Qual", "Neighborhood"],
    ["Neighborhood"],
    2,
    True,
    "Weinig clusters = grove indeling",
    "Te brede clusters",
    "Te weinig clusters mist detail"
)

run_experiment(
    "Exp 9",
    ["Gr Liv Area", "Overall Qual", "Year Built", "Full Bath", "Neighborhood"],
    ["Neighborhood"],
    4,
    True,
    "Balans tussen detail en structuur",
    "Redelijk stabiele clusters",
    "Goede middenweg"
)



Exp 1 - silhouette score: 0.12477010613899435
Cluster
0    2219
2     586
1     125
Name: count, dtype: int64
   feature_0  feature_1  feature_2  feature_3  feature_4  feature_5  \
0  -0.228675  -0.328551   0.018751   0.032589   0.062682   0.039563   
1   0.206507   0.170912  -0.058521  -0.101710  -0.195629  -0.123475   
2   0.821870   1.207664  -0.058521  -0.101710  -0.195629  -0.123475   

   feature_6  feature_7  feature_8  feature_9  ...  feature_19  feature_20  \
0   0.095193   0.051372   0.078071   0.078272  ...   -0.157588   -0.245067   
1  -0.316643  -0.190878  -0.266283  -0.244284  ...   -0.157588   -0.245067   
2  -0.292925  -0.153814  -0.238831  -0.244284  ...    0.630350    0.980268   

   feature_21  feature_22  feature_23  feature_24  feature_25  feature_26  \
0    0.090550    0.037801    0.074689   -0.211100   -0.257352   -0.133096   
1   -0.298018   -0.129055   -0.233101    4.737088   -0.257352   -0.133096   
2   -0.279314   -0.115611   -0.233101   -0.211100    1.029407

In [56]:
logboek.to_excel("logboek_cluster.xlsx", index=False)

print("Logboek opgeslagen!")


Logboek opgeslagen!
